# Does torch actually run on this GPU?

Two cells: what the install claims, then a small workload that proves it. A card can be
visible to `torch.cuda` and still refuse every kernel if its compute capability is newer
than anything the wheel was built for - which is why the second cell matters.

In [1]:
import torch

print(f"torch {torch.__version__}  (built against CUDA {torch.version.cuda})")
print(f"cuda available: {torch.cuda.is_available()}")
print(f"wheel was built for: {' '.join(torch.cuda.get_arch_list())}")

props = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info()
sm = f"sm_{props.major}{props.minor}"
print(f"device 0: {props.name}, {sm}, {free / 2**30:.2f} / {total / 2**30:.2f} GiB free")
print(f"{sm} in the wheel's arch list: {sm in torch.cuda.get_arch_list()}")

torch 2.13.0+cu132  (built against CUDA 13.2)
cuda available: True
wheel was built for: sm_75 sm_80 sm_86 sm_90 sm_100 sm_120
device 0: NVIDIA RTX PRO 4000 Blackwell, sm_120, 23.23 / 23.46 GiB free
sm_120 in the wheel's arch list: True


In [2]:
import time

N = 2048
try:
    a = torch.randn(N, N, device="cuda")
    b = torch.randn(N, N, device="cuda")
    (a @ b).sum().item()  # warm-up, and the first real kernel launch

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        c = a @ b
    torch.cuda.synchronize()
    secs = (time.perf_counter() - t0) / 10

    err = (c - (a.cpu() @ b.cpu()).cuda()).abs().max().item()
    print(f"OK - {N}x{N} matmul in {secs * 1000:.1f} ms "
          f"({2 * N**3 / secs / 1e12:.2f} TFLOP/s fp32), max err vs CPU {err:.2e}")

    print(f"     {torch.cuda.memory_reserved() / 2**20:.0f} MiB still reserved by torch")
except Exception as exc:
    print(f"FAILED - {type(exc).__name__}: {str(exc).splitlines()[0]}")

OK - 2048x2048 matmul in 0.8 ms (20.53 TFLOP/s fp32), max err vs CPU 1.49e-04
     114 MiB still reserved by torch


In [ ]:
# Idempotent: re-running this cell after the names are gone must not raise, otherwise
# the NameError aborts the cell and empty_cache() below never runs.
for name in ("a", "b", "c"):
    globals().pop(name, None)
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"torch reserved  {torch.cuda.memory_reserved() / 2**20:6.0f} MiB")
print(f"device in use   {(total - free) / 2**20:6.0f} MiB")
print("  of which ~600 MiB is the CUDA context and ~32 MiB the cuBLAS workspace -")
print("  neither is freeable from here, both go when the kernel is restarted.")